# Prac W5 - Performance

**Aims**:
- To gain some practical experience in evaluating supervised machine learning models

(**Question 1**) Repeat Q2 from Prac W2 10 times, saving the 10 resulting training and test sets

In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split 

from typing import List, Tuple

def load_data() -> pd.DataFrame:
    return pd.read_csv("./datasets/w3classif.csv", sep=",", header=None)

def get_train_test_splits(n: int, X: np.ndarray, y: np.array, test_size: float) -> Tuple(
    np.ndarray, np.ndarray, np.array, np.array):
    X_trains: List[np.ndarray] = []
    X_tests: List[np.ndarray] = []
    Y_trains: List[np.array] = []
    Y_tests: List[np.array] = []

    for i in range(0, n): 
        X_train, X_test, Y_train, Y_test = train_test_split(
            X, 
            y, 
            test_size=test_size, 
            random_state=i
        )

        X_trains.append(X_train)
        X_tests.append(X_test)
        Y_trains.append(Y_train)
        Y_tests.append(Y_test)

    return np.array(X_trains), np.array(X_tests), np.array(Y_trains), np.array(Y_tests)

df: pd.DataFrame = load_data()
X: np.ndarray = df.iloc[:, :2].to_numpy()
y: np.array = df.iloc[:, 2].to_numpy()

X_trains, X_tests, y_trains, y_tests = get_train_test_splits(10, X, y, 0.3)

(**Question 2**) Calculate the training and test set errors over all of the datasets from Q1 and 
calculate the average training and test errors over the 10 trials. Are the averages lower or higher
than the values you found in Prac W3?

In [10]:
from sklearn.pipeline import Pipeline, make_pipeline 
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

def build_knn_pipeline(k: int, X: np.ndarray, y: np.array) -> Pipeline:
    model: Pipeline = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(k)
    )

    model.fit(X, y)

    return model

def get_average_error(k: int, n: int, X_trains: np.ndarray, X_tests: np.ndarray, 
    y_trains: np.array, y_tests: np.array) -> Tuple(float, float):
    train_error_sum: float = 0.0
    test_error_sum: float = 0.0
    
    for i in range(0, 10):
        model: Pipeline = build_knn_pipeline(k, X_trains[i], y_trains[i])
        y_train_pred: np.array = model.predict(X_trains[i])
        y_test_pred: np.array = model.predict(X_tests[i])

        train_error_sum += 1 - accuracy_score(y_trains[i], y_train_pred)
        test_error_sum += 1 - accuracy_score(y_tests[i], y_test_pred)
    
    return train_error_sum / n, test_error_sum / n

average_train_error, average_test_error = get_average_error(
    3, 
    10, 
    X_trains, 
    X_tests,
    y_trains,
    y_tests    
)

print(f"Average Train Error: {average_train_error:.2f}")
print(f"Average Test Error: {average_test_error:.2f}")


Average Train Error: 0.02
Average Test Error: 0.05


The train average is lower than the value we gathered from Prac W2 (0.05) however the test average was higher (0.01)

(**Question 3**) Repeat Q1 and Q2 but use a different split - try 50/50 or 90/10. Compare your average error values with those you found in Q2

In [11]:
X_trains, X_tests, y_trains, y_tests = get_train_test_splits(10, X, y, 0.5)
average_train_error, average_test_error = get_average_error(
    3, 
    10, 
    X_trains, 
    X_tests, 
    y_trains, 
    y_tests
)

print("50/50 Split")
print(f"Average Train Error: {average_train_error:.2f}")
print(f"Average Test Error: {average_test_error:.2f}")

X_trains, X_tests, y_trains, y_tests = get_train_test_splits(10, X, y, 0.1)
average_train_error, average_test_error = get_average_error(
    3, 
    10, 
    X_trains, 
    X_tests, 
    y_trains, 
    y_tests
)

print("90/10 Split")
print(f"Average Train Error: {average_train_error:.2f}")
print(f"Average Test Error: {average_test_error:.2f}")

50/50 Split
Average Train Error: 0.03
Average Test Error: 0.05
90/10 Split
Average Train Error: 0.03
Average Test Error: 0.05


(**Question 4**) Calculate the sample standard deviation of your training and test set error values
over the 10 trials from Q2 and Q3. What do you observe?

In [13]:
def get_average_results(n:int, test_size: float) -> None:
    df: pd.DataFrame = load_data()
    X: np.ndarray = df.iloc[:, :2].to_numpy()
    y: np.array = df.iloc[:, 2].to_numpy()

    X_trains, X_tests, y_trains, y_tests = get_train_test_splits(
        n,
        X,
        y,
        test_size
    )

    train_errors_list: List[float] = []
    test_errors_list: List[float] = []
    
    for i in range(0, 10):
        model: Pipeline = build_knn_pipeline(3, X_trains[i], y_trains[i])
        y_train_pred: np.array = model.predict(X_trains[i])
        y_test_pred: np.array = model.predict(X_tests[i])

        train_errors_list.append(1 - accuracy_score(y_trains[i], y_train_pred))
        test_errors_list.append(1 - accuracy_score(y_tests[i], y_test_pred))

    train_errors: np.array = np.array(train_errors_list)
    test_errors: np.array = np.array(test_errors_list)

    print(f"Average Train Error: {np.mean(train_errors):.2f}")
    print(f"Train Sample Std: {np.std(train_errors):.2f}")
    print(f"Average Test Error: {np.mean(test_errors):.2f}")
    print(f"Test Sample Std: {np.std(test_errors):.2f}")

print("70/30 Split:")
get_average_results(10, 0.3)

print("50/50 Split")
get_average_results(10, 0.5)

print("90/10 Split")
get_average_results(10, 0.1)

70/30 Split:
Average Train Error: 0.02
Train Sample Std: 0.01
Average Test Error: 0.05
Test Sample Std: 0.02
50/50 Split
Average Train Error: 0.03
Train Sample Std: 0.01
Average Test Error: 0.05
Test Sample Std: 0.01
90/10 Split
Average Train Error: 0.03
Train Sample Std: 0.00
Average Test Error: 0.05
Test Sample Std: 0.04


(**Question 5**) Perform 10-fold cross validation using your model and the (original) dataset. 
What are the mean and standard deviations of the cross-validaton error?

In [15]:
from sklearn.model_selection import cross_val_score

model: Pipeline = build_knn_pipeline(3, X, y)
accuracy_scores = cross_val_score(model, X, y, cv=10, scoring='accuracy')
error_rates: np.array = 1 - accuracy_scores
print(f"CV Average Error: {np.mean(error_rates):.2f}")
print(f"CV Error Std: {np.std(error_rates):.2f}")

CV Average Error: 0.04
CV Error Std: 0.03
